# Project 10 — 3D Container Loading Optimizer

**3D Bin Packing** using three optimization algorithms:
Greedy Best-Fit Decreasing, Genetic Algorithm, and Simulated Annealing.

All three algorithms use **functionally equivalent shelf-based decoders**
with gravity enforcement and the **same fitness metrics** for scientifically
valid comparison.

---
## 1. Setup & Imports

In [ ]:
import sys, os, json, time, math, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Project root on path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'project10_container_dataset.json')
print(f'Project root: {PROJECT_ROOT}')
print(f'Data path:    {DATA_PATH}')

---
## 2. Dataset Exploration

In [ ]:
with open(DATA_PATH) as f:
    dataset = json.load(f)

cont = dataset['container']['inner_dimensions']
print(f"Container: {dataset['container']['type']}")
print(f"  Inner dims : {cont['length_cm']} x {cont['width_cm']} x {cont['height_cm']} cm")
container_vol = cont['length_cm'] * cont['width_cm'] * cont['height_cm']
print(f"  Volume     : {container_vol:,} cm3")
print(f"  Max payload: {dataset['container']['max_payload_kg']} kg\n")

print(f"{'Type':<40} {'LxWxH (cm)':<18} {'Weight':>8} {'Fragile':>8} {'Qty':>5}")
print('-' * 85)
total_boxes = total_vol = total_weight = 0
fragile_count = 0
for bt in dataset['box_types']:
    dims = f"{bt['length_cm']}x{bt['width_cm']}x{bt['height_cm']}"
    vol = bt['length_cm']*bt['width_cm']*bt['height_cm']*bt['quantity']
    total_vol += vol; total_boxes += bt['quantity']
    total_weight += bt['weight_kg']*bt['quantity']
    frag = 'Yes' if bt.get('fragile', False) else ''
    if bt.get('fragile', False): fragile_count += bt['quantity']
    print(f"{bt['label']:<40} {dims:<18} {bt['weight_kg']:>7.1f} {frag:>8} {bt['quantity']:>5}")
print('-' * 85)
print(f"{'TOTAL':<40} {'':>18} {total_weight:>7.1f} {'':>8} {total_boxes:>5}")
print(f"\nFragile boxes: {fragile_count} / {total_boxes}")
print(f"Theoretical max utilization: {total_vol/container_vol*100:.2f}%")

### Box Type Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

types = dataset['box_types']
labels = [t['label'].split('(')[0].strip()[:18] for t in types]
volumes = [t['length_cm']*t['width_cm']*t['height_cm'] for t in types]
weights = [t['weight_kg'] for t in types]
quantities = [t['quantity'] for t in types]

axes[0].barh(labels, volumes, color='#3498DB', alpha=0.85)
axes[0].set_xlabel('Volume (cm3)'); axes[0].set_title('Box Volume by Type', fontweight='bold')
axes[0].invert_yaxis()

axes[1].barh(labels, weights, color='#E74C3C', alpha=0.85)
axes[1].set_xlabel('Weight (kg)'); axes[1].set_title('Box Weight by Type', fontweight='bold')
axes[1].invert_yaxis()

axes[2].barh(labels, quantities, color='#2ECC71', alpha=0.85)
axes[2].set_xlabel('Quantity'); axes[2].set_title('Box Quantity by Type', fontweight='bold')
axes[2].invert_yaxis()

plt.tight_layout(); plt.show()

---
## 3. Core Modules

The project's shared infrastructure lives in `core/`:

| Module | Purpose |
|--------|--------|
| `models.py` | `Box` and `Container` data classes |
| `data_loader.py` | JSON dataset parser with fragile/orientation-locked support |
| `utils.py` | Unified utilization & stability metrics (used by all algorithms) |
| `space_manager.py` | Shelf-based placement decoder with gravity drop |

All three algorithms use **functionally equivalent shelf-based decoders**:

| Algorithm | Decoder Function | Gravity |
|-----------|-----------------|--------|
| Greedy | `space_manager.place_boxes()` | `_gravity_drop()` built-in |
| GA | `GeneticAlgorithm._decode_chromosome()` | `_gravity_settle()` |
| SA | `SimulatedAnnealing.decode()` | `_gravity_drop()` |

They share the same fitness formula: `utilization + 0.20 * stability + placed_bonus - 35.0 * unplaced`

In [ ]:
from core.data_loader import load_data
from core.models import Box, Container
from core.utils import compute_utilization, compute_stability
from core.space_manager import place_boxes

boxes, container = load_data(DATA_PATH)
print(f'Loaded {len(boxes)} boxes')
print(f'Container: {container.L} x {container.W} x {container.H} cm')
print(f'Container volume: {container.volume():,} cm3')
print(f'Fragile boxes: {sum(1 for b in boxes if b.fragile)} / {len(boxes)}')
total_box_vol = sum(b.volume() for b in boxes)
print(f'Total box volume: {total_box_vol:,} cm3')
print(f'Theoretical max utilization: {total_box_vol / container.volume() * 100:.2f}%')

---
## 4. Algorithm 1 — Greedy Best-Fit Decreasing (BFD)

### Strategy
1. **Try 12 heuristic orderings** (volume, height, base area, weight, etc.)
2. **Place** each ordering through the shelf-based decoder (`space_manager.place_boxes`)
3. **Keep** the ordering that yields the highest utilization

Gravity is enforced by the `_gravity_drop()` pass inside `place_boxes()`.

| Strengths | Weaknesses |
|-----------|------------|
| Very fast (< 0.1s) | No search — fixed heuristics only |
| Deterministic, reproducible | Cannot escape local optima |
| Good baseline for comparison | No rotation optimization per box |

In [ ]:
from algorithms.greedy import GreedyBFD

start = time.perf_counter()
greedy_result = GreedyBFD.solve(boxes, container)
greedy_time = time.perf_counter() - start
greedy_positions = greedy_result['positions']
greedy_util = greedy_result['utilization']

print('Greedy BFD Results')
print(f'  Boxes placed : {len(greedy_positions)} / {len(boxes)}')
print(f'  Utilization  : {greedy_util:.2f}%')
print(f'  Time         : {greedy_time:.3f}s')

---
## 5. Algorithm 2 — Genetic Algorithm (GA)

### Key Components
| Component | Implementation |
|-----------|----------------|
| **Chromosome** | Permutation of (box_index, rotation_id) tuples |
| **Decoder** | Internal shelf-based placer (`_decode_chromosome`) + `_gravity_settle` |
| **Fitness** | utilization + 0.20 x stability + placed_bonus - 35.0 x unplaced |
| **Selection** | Tournament (size 5) |
| **Crossover** | OX1 (45%), OX2 (35%), PMX (20%) — proper mapping-table |
| **Mutation** | Swap, Inversion, Rotation, Scramble (adaptive pressure) |
| **Elitism** | Top 4 survive |
| **Seeding** | 12 heuristic orderings for initial population |
| **Stagnation** | Population restart after 20 gen without improvement |
| **Early stop** | Terminates if utilization >= 80% with all boxes placed |

In [ ]:
from algorithms.genetic import GeneticAlgorithm

# Use reduced parameters for notebook responsiveness
GeneticAlgorithm.POPULATION_SIZE = 30
GeneticAlgorithm.NUM_GENERATIONS = 40

start = time.perf_counter()
ga_result = GeneticAlgorithm.solve(boxes, container, seed=42)
ga_time = time.perf_counter() - start
ga_positions = ga_result['positions']
ga_util = ga_result['utilization']

print('Genetic Algorithm Results')
print(f'  Boxes placed : {len(ga_positions)} / {len(boxes)}')
print(f'  Utilization  : {ga_util:.2f}%')
print(f'  Best fitness : {ga_result["fitness"]:.2f}')
print(f'  Time         : {ga_time:.2f}s')

### GA Convergence Plot

In [ ]:
history = ga_result.get('history', [])
if history:
    plt.figure(figsize=(10, 4))
    plt.plot(history, color='#E74C3C', linewidth=2, alpha=0.9)
    plt.fill_between(range(len(history)), history, alpha=0.1, color='#E74C3C')
    plt.xlabel('Generation', fontsize=12)
    plt.ylabel('Best Utilization (%)', fontsize=12)
    plt.title('GA Convergence — Best Utilization Over Generations',
              fontsize=14, fontweight='bold')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No convergence history available.')

---
## 6. Algorithm 3 — Simulated Annealing (SA)

### Key Components
| Component | Implementation |
|-----------|----------------|
| **Solution** | Permutation of box indices |
| **Decoder** | Internal shelf-based placer (`decode`) + `_gravity_drop` |
| **Fitness** | utilization + 0.20 x stability + placed_bonus - 35.0 x unplaced (same as GA) |
| **Temperature** | T0=10, alpha=0.99954 (calibrated for ~40% initial acceptance) |
| **Perturbation** | Swap (35%), Reverse segment (30%), Insert (35%) |
| **Acceptance** | Metropolis criterion: P(accept) = exp(-delta_f / T) |
| **Restart** | On stagnation (3000 iters), restart from global best |
| **Time limit** | Wall-clock cap (60s in notebook, 300s in full run) |

In [ ]:
from algorithms.simulated_annealing import SimulatedAnnealing

# Use 60s time limit for notebook responsiveness (full run uses 300s)
start = time.perf_counter()
sa_result = SimulatedAnnealing.solve(boxes, container, time_limit=60.0, seed=42)
sa_time = time.perf_counter() - start
sa_positions = sa_result['positions']
sa_util = sa_result['utilization']

print('Simulated Annealing Results')
print(f'  Boxes placed : {len(sa_positions)} / {len(boxes)}')
print(f'  Utilization  : {sa_util:.2f}%')
print(f'  Fitness      : {sa_result["fitness"]:.2f}')
print(f'  Time         : {sa_time:.2f}s')

---
## 7. Constraint Validation

We verify all solutions satisfy three physical constraints:
1. **No overlaps** — no two boxes interpenetrate
2. **No boundary violations** — all boxes fit within container dimensions
3. **Gravity** — every box rests on the floor or on another box (no floating)

In [ ]:
def validate_solution(positions, container, label, dim_keys=('l','w','h')):
    lk, wk, hk = dim_keys
    CL, CW, CH = container.L, container.W, container.H
    errors = {'overlap': 0, 'boundary': 0, 'floating': 0}
    for p in positions:
        if (p['x'] < -0.01 or p['y'] < -0.01 or p['z'] < -0.01 or
            p['x'] + p[lk] > CL + 1 or p['y'] + p[wk] > CW + 1 or p['z'] + p[hk] > CH + 1):
            errors['boundary'] += 1
    for i, a in enumerate(positions):
        for j, b in enumerate(positions):
            if j <= i: continue
            eps = 0.5
            if not (a['x']+a[lk] <= b['x']+eps or b['x']+b[lk] <= a['x']+eps or
                    a['y']+a[wk] <= b['y']+eps or b['y']+b[wk] <= a['y']+eps or
                    a['z']+a[hk] <= b['z']+eps or b['z']+b[hk] <= a['z']+eps):
                errors['overlap'] += 1
    for p in positions:
        if p['z'] < 0.01: continue
        supported = False
        for q in positions:
            if q is p: continue
            if abs((q['z'] + q[hk]) - p['z']) < 1.0:
                if (p['x'] < q['x']+q[lk]-0.01 and p['x']+p[lk] > q['x']+0.01 and
                    p['y'] < q['y']+q[wk]-0.01 and p['y']+p[wk] > q['y']+0.01):
                    supported = True; break
        if not supported: errors['floating'] += 1
    status = 'PASS' if all(v == 0 for v in errors.values()) else 'FAIL'
    print(f'[{label}] {status}  --  Overlaps: {errors["overlap"]}, '
          f'Boundary: {errors["boundary"]}, Floating: {errors["floating"]}')
    return errors

print('Constraint Validation Results')
print('=' * 70)
validate_solution(greedy_positions, container, 'Greedy')
validate_solution(ga_positions, container, 'GA    ')
validate_solution(sa_positions, container, 'SA    ')

---
## 8. Comparative Analysis

In [ ]:
print(f"{'Algorithm':<22} {'Utilization':>12} {'Placed':>10} {'Time':>10}")
print('=' * 58)
for name, util, placed, t in [
    ('Greedy BFD', greedy_util, len(greedy_positions), greedy_time),
    ('Genetic Algorithm', ga_util, len(ga_positions), ga_time),
    ('Simulated Annealing', sa_util, len(sa_positions), sa_time)]:
    print(f'{name:<22} {util:>11.2f}% {placed:>7}/{len(boxes)} {t:>9.2f}s')

print()
print('Notes:')
print('  - All algorithms use functionally equivalent shelf-based decoders')
print('  - All algorithms use the same stability formula (core.utils.compute_stability)')
print('  - All algorithms use the same fitness weights (stability=0.20, penalty=35.0/box)')
print('  - GA uses pop=30, gen=40 (notebook); full run uses pop=50, gen=100')
print('  - SA uses time_limit=60s (notebook); full run uses 300s')
print('  - Both GA and SA use random.seed(42) for reproducibility')

### Utilization Comparison Chart

In [ ]:
algos = ['Greedy BFD', 'Genetic Algorithm', 'Simulated Annealing']
utils_list = [greedy_util, ga_util, sa_util]
colors = ['#3498DB', '#E74C3C', '#2ECC71']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(algos, utils_list, color=colors, width=0.5, zorder=3)
for bar, val in zip(bars, utils_list):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Utilization (%)')
axes[0].set_title('Volume Utilization', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].axhline(y=80.96, color='gray', linestyle='--', alpha=0.5, label='Theoretical Max')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3, zorder=0)

placed_counts = [len(greedy_positions), len(ga_positions), len(sa_positions)]
unplaced = [len(boxes)-p for p in placed_counts]
bars_p = axes[1].bar(algos, placed_counts, color=colors, width=0.5, label='Placed', zorder=3)
axes[1].bar(algos, unplaced, bottom=placed_counts, color=['#BDC3C7']*3,
            width=0.5, label='Unplaced', zorder=3, alpha=0.6)
for bar, val in zip(bars_p, placed_counts):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()/2, str(val),
                ha='center', va='center', fontweight='bold', fontsize=14, color='white')
axes[1].set_ylabel('Boxes')
axes[1].set_title('Boxes Placed vs Unplaced', fontsize=14, fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout(); plt.show()

### Strengths & Weaknesses

| Algorithm | Strengths | Weaknesses |
|-----------|-----------|------------|
| **Greedy BFD** | Extremely fast (<0.1s), deterministic, good baseline | No search — fixed orderings only |
| **Genetic Algorithm** | Multi-objective fitness, adaptive mutation, diverse operators | Stochastic, slower (~80s full run) |
| **Simulated Annealing** | Escapes local optima, time-bounded, multi-restart | Stochastic, slowest (~5 min full run) |

---
## 9. 3D Visualization

Below we render each algorithm's packing result inside the container wireframe.
Boxes are color-coded and semi-transparent so stacking is visible.

In [ ]:
def draw_container_wireframe(ax, CL, CW, CH):
    edges = [
        ([0,CL],[0,0],[0,0]),  ([0,CL],[CW,CW],[0,0]),
        ([0,CL],[0,0],[CH,CH]),([0,CL],[CW,CW],[CH,CH]),
        ([0,0],[0,CW],[0,0]),  ([CL,CL],[0,CW],[0,0]),
        ([0,0],[0,CW],[CH,CH]),([CL,CL],[0,CW],[CH,CH]),
        ([0,0],[0,0],[0,CH]),  ([CL,CL],[0,0],[0,CH]),
        ([0,0],[CW,CW],[0,CH]),([CL,CL],[CW,CW],[0,CH]),
    ]
    for xs, ys, zs in edges:
        ax.plot3D(xs, ys, zs, color='#555555', linewidth=1.2,
                  linestyle='--', alpha=0.6)

def draw_box_3d(ax, x, y, z, dx, dy, dz, color='#3498DB', alpha=0.3):
    verts = [
        [[x,y,z],[x+dx,y,z],[x+dx,y+dy,z],[x,y+dy,z]],
        [[x,y,z+dz],[x+dx,y,z+dz],[x+dx,y+dy,z+dz],[x,y+dy,z+dz]],
        [[x,y,z],[x+dx,y,z],[x+dx,y,z+dz],[x,y,z+dz]],
        [[x,y+dy,z],[x+dx,y+dy,z],[x+dx,y+dy,z+dz],[x,y+dy,z+dz]],
        [[x,y,z],[x,y+dy,z],[x,y+dy,z+dz],[x,y,z+dz]],
        [[x+dx,y,z],[x+dx,y+dy,z],[x+dx,y+dy,z+dz],[x+dx,y,z+dz]],
    ]
    ax.add_collection3d(Poly3DCollection(
        verts, alpha=alpha, facecolor=color,
        edgecolor='#1a1a2e', linewidth=0.4))

### Side-by-Side Overview

In [ ]:
fig = plt.figure(figsize=(18, 6))
all_results = [
    ('Greedy BFD', greedy_positions, greedy_util, '#3498DB'),
    ('Genetic Algorithm', ga_positions, ga_util, '#E74C3C'),
    ('Simulated Annealing', sa_positions, sa_util, '#2ECC71'),
]
CL, CW, CH = container.L, container.W, container.H

for idx, (name, positions, util, base_color) in enumerate(all_results):
    ax = fig.add_subplot(1, 3, idx+1, projection='3d')
    ax.set_facecolor('#f8f9fa')
    draw_container_wireframe(ax, CL, CW, CH)

    subset = positions[:60]
    cmap = plt.cm.tab20
    for i, p in enumerate(subset):
        draw_box_3d(ax, p['x'], p['y'], p['z'],
                    p['l'], p['w'], p['h'],
                    color=cmap(i % 20), alpha=0.35)

    ax.set_xlim(0, CL); ax.set_ylim(0, CW); ax.set_zlim(0, CH)
    ax.set_xlabel('L'); ax.set_ylabel('W'); ax.set_zlabel('H')
    ax.set_title(f'{name}\n{util:.1f}% | {len(positions)} boxes',
                 fontsize=11, fontweight='bold')
    ax.view_init(elev=22, azim=-55)

plt.suptitle('3D Container Packing - All Algorithms',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 9.2 Web Application Architecture

```
Flask API (api/main.py)     <--->     Three.js Frontend (visualization/)
  POST /api/solve                     Interactive 3D container rendering
  POST /api/compare                   with orbit controls
  GET  /api/dataset                   (includes fragile/orientation_locked)
```

Run: `python api/main.py` then open `http://localhost:5000`

---
## 10. Conclusion

### Summary
All three algorithms use **functionally equivalent shelf-based decoders**
and the **same fitness metrics**, enabling valid comparison:
- **Greedy** provides a fast, deterministic baseline (~76% utilization in <0.1s)
- **GA** uses evolutionary search with diverse operators for multi-objective optimization
- **SA** offers time-bounded probabilistic search with multi-restart

### Physical Validity
All solutions pass constraint validation:
**no overlaps, no boundary violations, full gravity compliance.**
Gravity is enforced via `_gravity_drop()` / `_gravity_settle()` post-processing
in each decoder.

### Unified Design
- Decoders: shelf-based, O(n) per call, functionally equivalent across algorithms
- Stability metric: `core.utils.compute_stability()` (weight x height formula)
- Fitness: `utilization + 0.20 * stability + placed_bonus - 35.0 * unplaced`
- Fragile/orientation-locked: checked consistently across all modules
- Reproducibility: `random.seed(42)` in both GA and SA

### Future Improvements
1. Hybrid: Greedy seed -> GA refinement -> SA fine-tuning
2. Heightmap-based DBLF for tighter packing within layers
3. Multi-container load balancing
4. Parallel GA evaluation for speed

---